SECTION 0 — Setup


In [1]:
# Section 0 — Setup
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
 
sys.path.insert(0, str(Path.cwd().parent))
 
from src.config import (
    TRAIN_TEST_DIR, NFHS5_CLEANED_PATH, TARGET_COLS,
    FIGURES_DIR, TABLES_DIR, MODELS_DIR,
)
from src.model import load_model
from src.explainability import SHAPExplainer
from src.fairness import FairnessAuditor, SENSITIVE_FEATURES, FNR_TOLERANCE
from src.preprocessing import detect_sc_st_column
from src.logger import get_console_logger
 
sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
log = get_console_logger('03_shap_fairness')
print('Setup complete')
 


Setup complete


In [2]:
# Load model and splits
mltp = load_model(MODELS_DIR / 'mltp_xgb_v1.pkl')
 
X_train = pd.read_csv(TRAIN_TEST_DIR / 'X_train.csv')
X_test  = pd.read_csv(TRAIN_TEST_DIR / 'X_test.csv')
y_test  = pd.read_csv(TRAIN_TEST_DIR / 'y_test.csv')
print(f'Model loaded: mltp_xgb_v1.pkl')
print(f'X_test: {X_test.shape}')
 


2026-05-12 16:41:34  INFO      src.model: Model loaded: /workspaces/malnutrisense/models/mltp_xgb_v1.pkl
Model loaded: mltp_xgb_v1.pkl
X_test: (39760, 11)


In [4]:
# Load full cleaned dataset to get demographic columns for fairness audit
df_full = pd.read_csv(NFHS5_CLEANED_PATH)
 
# Align demographic columns with X_test index
# X_test was produced by train_test_split with index preserved from df_full
df_meta = df_full.loc[X_test.index].reset_index(drop=True)
 
# Add is_aspirational flag if not already present
if 'is_aspirational' not in df_meta.columns:
    asp = pd.read_csv('/workspaces/malnutrisense/data/raw/external/aspirational_districts.csv')
    df_meta['is_aspirational'] = 0   # default 0, update via district merge if available
 
# Detect SC/ST column
sc_st_col = detect_sc_st_column(df_meta)
if sc_st_col:
    from src.fairness import SENSITIVE_FEATURES
    SENSITIVE_FEATURES['sc_st_group'] = sc_st_col
    print(f'SC/ST column: {sc_st_col}')
else:
    print('SC/ST column not found — ST subgroup audit skipped')
print(f'df_meta: {df_meta.shape}')
 


2026-05-12 16:44:30  WARNING   src.preprocessing: No SC/ST column found. Searched: ['V131', 'S116', 'SH46', 'V130']. Fairness audit for Scheduled Tribe subgroup will be unavailable. Consider requesting the Household Members Recode (PR) file from DHS.
SC/ST column not found — ST subgroup audit skipped
df_meta: (39760, 18)


SECTION 1 — SHAP Analysis 

In [5]:
# Section 1 — Initialise SHAPExplainer with 200-row background sample
bg_sample = X_train.sample(200, random_state=42)
explainer = SHAPExplainer(mltp, bg_sample)
print(f'Explainer ready. Features: {len(explainer.feature_names)}')


2026-05-12 16:45:16  INFO      src.explainability: SHAPExplainer initialised for 3 labels
Explainer ready. Features: 11


In [6]:
# Compute SHAP values for entire test set
# This takes 30-120 seconds depending on test set size
print('Computing SHAP values...')
shap_values = explainer.compute_shap_values(X_test)
print(f'SHAP values computed: {shap_values[0].shape}')
 


Computing SHAP values...


100%|===================| 39749/39760 [08:22<00:00]        

2026-05-12 17:10:06  INFO      src.explainability: SHAP values computed: 39760 rows, 11 features
SHAP values computed: (39760, 11)


In [7]:
# Global feature importance — one summary plot per label
for label in TARGET_COLS:
    path = explainer.plot_summary(shap_values, X_test, label, max_features=15)
    print(f'Saved: {path}')
 
# Display the stunted summary inline
from IPython.display import Image
Image(filename=str(explainer.feature_names), width=700)  # placeholder — adjust path


TypeError: summary_legacy() got an unexpected keyword argument 'ax'